# The Z boson peak from CMS open data

This notebook reconstructs the Z → μ⁺μ⁻ peak from real collisions. It runs from a fresh clone on the small sample in
`examples/data/`, with no large download. The last part reads the results of the production run.

In [ ]:
import json
import os
import re
from pathlib import Path

from IPython.display import Image, Markdown, display

import cmsopen
from cmsopen.fitting import pdg_mass_width
from cmsopen.pipeline import CHANNELS

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "data" / "cmsopen_sample.root").exists())
SAMPLE = ROOT / "examples" / "data" / "cmsopen_sample.root"
FIGURES = ROOT / "docs" / "figures"
PRODUCTION = json.loads((ROOT / "results" / "production.json").read_text())
print(f"cmsopen {cmsopen.__version__} imported from: {cmsopen.__file__}")


class Checks:
    """Collects self-check results and prints one OK/FAIL line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail=""):
        """Print one OK or FAIL line and remember failures."""
        print(f"{'OK' if ok else 'FAIL'}: {name}" + (f" ({detail})" if detail else ""))
        if not ok:
            self.failures.append(name)


checks = Checks()

## Why a peak?

A Z boson decays before it can reach a detector, but energy and momentum are conserved, so the two muons together have
the invariant mass of the Z. Muon pairs from other sources have no preferred mass and form a smooth background. A
histogram of the pair mass is therefore a peak on a smooth continuum.

## Analyse the sample

The sample holds the first events of one file of Run 2012C, with only the branches the analysis reads. It has few
pairs, so here we fit `voigt_poly`, a simpler line shape with one free width. The numbers this cell prints are a
demonstration and are not results.

In [ ]:
result = cmsopen.analyze(SAMPLE, channel="Z_mumu", model="voigt_poly", stability=False, verbose=False)
print(result.fit.table())
display(result.figures["mass_fit"])

In [ ]:
pdg_mass = pdg_mass_width(CHANNELS["Z_mumu"].resonance)[0]
peak, error = result.fit.peak_position
n_pairs = result.fit.n_entries
checks.check("the sample gives a fit with pairs in its range", n_pairs > 500, f"{n_pairs:,} pairs")
checks.check("the peak is within a generous window of the PDG mass", abs(peak - pdg_mass) < 1.0, f"{peak:.3f} GeV")
checks.check("the figures exist", {"mass_fit", "cutflow", "os_vs_ss"} <= set(result.figures))

## Only opposite-sign pairs make a peak

A neutral parent decays to one μ⁺ and one μ⁻. Same-sign pairs cannot come from a Z, so they show what the background
looks like.

In [ ]:
display(result.figures["os_vs_ss"])

## Optional: a whole file

If `CMSOPEN_DATA_FILE` points to a NanoAOD file, this cell runs the frozen default model on it (a minute or so per
file). Without it, the cell does nothing.

In [ ]:
data_file = os.environ.get("CMSOPEN_DATA_FILE")
if data_file:
    full = cmsopen.analyze(data_file, channel="Z_mumu", stability=False, verbose=False)
    print(full.fit.table())
    display(full.figures["mass_fit"])
else:
    print("CMSOPEN_DATA_FILE is not set: skipped")

## Production results

These come from the single run on the pre-registered sample, not from anything computed above. A benchmark is a requirement fixed
before the sample was opened; the observed values are what was measured.



Measured on 20 files (controlling **benchmark**: 20 pre-registered files, met).

| Z → μ⁺μ⁻ | value |
|---|---|
| Model and fit range | `cb2_bw_exp`, 70-110 GeV |
| Events read, pairs in the fit range | 9,780,165, 584,716 |
| Peak (pole) position, observed | 90.9000 GeV |
| Statistical error, raw | 0.0050 GeV |
| Statistical error, scaled by S = 1.232 | 0.0061 GeV |
| Resolution, observed | 1.9821 ± 0.0075 GeV |
| χ²/ndf **benchmark** | ≤ 2.0 |
| χ²/ndf, observed | 48.6 / 32 = 1.518 (met) |
| PDG mass (reference only) | 91.1879 GeV |
| Offset from PDG, observed | -0.2879 GeV |
| Fit-choice spread, observed | 0.0028 GeV |

The peak sits below the PDG mass. No muon momentum-scale corrections are applied, so this is not a measurement of
the Z mass. The scale factor S is the square root of χ²/ndf, floored at one.

In [ ]:
NOTEBOOK = "01_z_peak.ipynb"
NOTEBOOK_CHANNELS = ("Z_mumu",)
NOTEBOOK_FIGURES = ("z_peak_fit.png", "z_os_vs_ss.png", "cutflow.png", "muon_pt.png", "muon_eta.png")
# Structural self-checks: every number stated in the "Production results" text equals results/production.json.
def gev(value, channel, sign=False):
    """A GeV value with the channel's decimals, as in the text."""
    decimals = 4 if channel == "Z_mumu" else 5
    if value is None:
        return "n/a"
    return f"{value:+.{decimals}f}" if sign else f"{value:.{decimals}f}"


def stated_numbers(channel):
    """The text's numbers for one channel, recomputed here from the production file."""
    c = PRODUCTION["channels"][channel]
    r = c["result"]
    numbers = {
        "peak position": gev(r["peak_position_gev"], channel),
        "raw stat. error": gev(r["stat_error_gev"], channel),
        "scaled stat. error": gev(r["stat_error_scaled_gev"], channel),
        "scale factor": f"{r['scale_factor']:.3f}",
        "resolution": gev(r["resolution_gev"], channel),
        "resolution error": gev(r["resolution_error_gev"], channel),
        "chi2": f"{r['chi2']:.1f}",
        "ndf": str(r["ndf"]),
        "chi2/ndf": f"{r['chi2_ndf']:.3f}",
        "chi2/ndf benchmark": f"{c['t2']['benchmark']:.1f}",
        "PDG mass": f"{r['pdg_mass_gev']:.4f}",
        "offset": gev(r["offset_from_pdg_gev"], channel, sign=True),
        "fit-choice spread": gev(c["fit_choice_spread_gev"], channel),
        "pairs": f"{r['n_pairs_in_fit_range']:,}",
        "events": f"{r['n_events']:,}",
        "fit range low": f"{r['fit_range_gev'][0]:g}",
        "fit range high": f"{r['fit_range_gev'][1]:g}",
        "files": str(PRODUCTION["sample"]["n_files"]),
        "benchmark files": str(PRODUCTION["targets"]["T4"]["benchmark"]["n_files"]),
    }
    return {f"{channel} {name}": value for name, value in numbers.items()}


notebook = json.loads((ROOT / "notebooks" / NOTEBOOK).read_text())
production_cells = [cell for cell in notebook["cells"] if "production-text" in cell["metadata"].get("tags", [])]
text = "\n".join("".join(cell["source"]) for cell in production_cells)
stated = {}
for channel in NOTEBOOK_CHANNELS:
    stated.update(stated_numbers(channel))
for name, value in stated.items():
    checks.check(f"the text states the {name} of production.json", value.lstrip("+") in text, value)
visible = re.sub(r"\]\([^)]*\)", "]", text)
visible = re.sub(r"<!--.*?-->", "", visible, flags=re.S).replace("Run 2012", "").replace("8 TeV", "")
found = re.findall(r"(?<![\w.])\d[\d,]*(?:\.\d+)?(?![\w])", visible)
known = {value.lstrip("+-") for value in stated.values()}
typed = sorted({n.rstrip(",") for n in found} - known)
checks.check("every number in the text comes from production.json", not typed, ", ".join(typed) or "none other")
for name in NOTEBOOK_FIGURES:
    path = FIGURES / name
    checks.check(f"docs/figures/{name} exists and is listed in production.json",
                 path.exists() and f"docs/figures/{name}" in PRODUCTION["figures"])

In [ ]:
display(Image(filename=str(FIGURES / "z_peak_fit.png"), width=720))
display(Markdown((FIGURES / "z_peak_fit_caption.md").read_text()))

The fit's lower panel shows the pulls, (data − fit) / σ per bin. If the model describes the shape, they show no pattern.

In [ ]:
display(Image(filename=str(FIGURES / "z_os_vs_ss.png"), width=720))
display(Image(filename=str(FIGURES / "cutflow.png"), width=720))

The first figure compares opposite-sign and same-sign pairs: a neutral parent can only give opposite-sign ones. The second shows how many events survive each selection step.

In [ ]:
if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")